# EvoVariant-TR autonomous adaptation runner

**Study:** `POSTHOC-FOUNDATION-ADAPTATION-001` · **Protocol SHA-256:** `07c93b4657e84a4ddfbdc2df1af0f467f80959e0534a67840b4bf2b2b04a2c2c`.

Run cells in order. The runner verifies state and artifacts, skips completed stages, and uses persisted Drive checkpoints/SQLite to resume incomplete Caduceus work. It never runs all cells automatically. Only free Colab GPU is used; VALIDATION stays closed until the selection lock exists; the 946-row locked test is never loaded.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time

ROOT = Path('/content/EvoVariant')
DRIVE_ROOT = Path('/content/drive/MyDrive/EvoVariantTR')
BRANCH = 'research/posthoc-foundation-adaptation'
STATE = DRIVE_ROOT / 'state' / 'adaptation_state.json'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for name in ('reference','datasets','checkpoints','model_cache','runs','hpo','logs','state','exports'):
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)


In [ ]:
if not ROOT.exists():
    subprocess.run(['git','clone','-b',BRANCH,'https://github.com/UtkarsHMer05/EvoVariant-TR-.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin',BRANCH],check=True)
head = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
branch = subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'],text=True).strip()
status = subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain'],text=True).strip()
assert branch == BRANCH and not status, (branch, status)
print('branch=',branch,'HEAD=',head,'worktree=clean')
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')


In [ ]:
ENV = Path('/content/caduceus-env')
UV = shutil.which('uv') or '/usr/local/bin/uv'
if not shutil.which('uv') and not Path(UV).exists():
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    UV = shutil.which('uv') or '/usr/local/bin/uv'
if not ENV.exists():
    subprocess.run([UV,'venv','--python','3.11',str(ENV)],check=True)
PY = str(ENV / 'bin' / 'python')
probe = subprocess.run([PY,'-c','import torch,transformers,mamba_ssm; print(torch.__version__,torch.cuda.is_available(),torch.cuda.get_device_name(0),transformers.__version__)'],text=True,capture_output=True)
if probe.returncode:
    subprocess.run([UV,'pip','install','--python',PY,'--index-url','https://download.pytorch.org/whl/cu121','torch==2.2.0'],check=True)
    subprocess.run([UV,'pip','install','--python',PY,'--no-build-isolation','-r',str(ROOT/'requirements-adaptation.txt')],check=True)
print(subprocess.check_output([PY,'-c','import numpy,torch,transformers,mamba_ssm; print("numpy",numpy.__version__,"torch",torch.__version__,"cuda",torch.cuda.is_available(),torch.cuda.get_device_name(0),"transformers",transformers.__version__)'],text=True))


In [ ]:
ARCHIVE = DRIVE_ROOT/'reference'/'hg38.fa.gz'
REFERENCE = Path('/content/Homo_sapiens_assembly38.fasta')
EXPECTED_ARCHIVE_BYTES = 983659424
EXPECTED_REFERENCE_SHA256 = '5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51'
if not ARCHIVE.exists() or ARCHIVE.stat().st_size != EXPECTED_ARCHIVE_BYTES:
    url='https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz'
    subprocess.run(['curl','-fL','--retry','3','--connect-timeout','20','--max-time','1800','--resolve','hgdownload.soe.ucsc.edu:443:128.114.119.163','-o',str(ARCHIVE),url],check=True)

def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(8*1024*1024),b''):
            digest.update(chunk)
    return digest.hexdigest()

if not REFERENCE.exists() or sha256(REFERENCE) != EXPECTED_REFERENCE_SHA256:
    import gzip
    digest=hashlib.sha256()
    with gzip.open(ARCHIVE,'rb') as source, REFERENCE.open('wb') as target:
        for chunk in iter(lambda:source.read(8*1024*1024),b''):
            target.write(chunk); digest.update(chunk)
    if digest.hexdigest() != EXPECTED_REFERENCE_SHA256:
        raise RuntimeError(f'GRCh38 FASTA hash mismatch: {digest.hexdigest()}')
print('GRCh38 FASTA verified:',REFERENCE.stat().st_size,'bytes',sha256(REFERENCE))


In [ ]:
ENV_REPORT = DRIVE_ROOT/'state'/'environment.json'
MANIFEST_REPORT = DRIVE_ROOT/'state'/'manifests_verified.json'
DATA_REPORT = DRIVE_ROOT/'state'/'data_ready.json'
subprocess.run([PY,'scripts/adaptation/hardware_probe.py','--root',str(ROOT),'--output',str(ENV_REPORT),'--state',str(STATE)],check=True)
subprocess.run([PY,'scripts/adaptation/verify_data.py','--root',str(ROOT),'--output',str(MANIFEST_REPORT),'--state',str(STATE)],check=True)
subprocess.run([PY,'scripts/adaptation/verify_data.py','--root',str(ROOT),'--reference',str(REFERENCE),'--output',str(DATA_REPORT),'--state',str(STATE)],check=True)
ready=json.loads(DATA_REPORT.read_text())
assert ready['reference']['sha256'] == EXPECTED_REFERENCE_SHA256 and ready['reference']['checked_formal_ref_alleles'] == 4000
shutil.copy2(REFERENCE.with_suffix('.fasta.fai'),DRIVE_ROOT/'reference'/'Homo_sapiens_assembly38.fasta.fai')
print(DATA_REPORT.read_text())


In [ ]:
SMOKE = DRIVE_ROOT/'runs'/'caduceus_smoke.json'
SMOKE_CKPT = DRIVE_ROOT/'checkpoints'/'caduceus_smoke.pt'
if SMOKE.exists() and SMOKE_CKPT.exists() and json.loads(SMOKE.read_text()).get('status') == 'PASS':
    print('Caduceus smoke already passed:',json.loads(SMOKE.read_text()))
else:
    subprocess.run([PY,'scripts/adaptation/smoke_caduceus.py','--root',str(ROOT),'--device','cuda','--cache-dir',str(DRIVE_ROOT/'model_cache'),'--output',str(SMOKE),'--checkpoint',str(SMOKE_CKPT),'--state',str(STATE)],check=True)


## Start or resume the TRAIN-only frozen-head baseline

This job trains on all 3,199 TRAIN rows at 8,192 bp, with frozen encoder/head trainable, seed 42. It writes an atomic Drive checkpoint at every epoch boundary. If the Colab kernel restarts, rerun setup cells, then this launcher; it resumes from `latest.pt` or detects a still-live process.


In [ ]:
def launch(stage, command, report, log_path):
    report=Path(report); log_path=Path(log_path)
    if report.exists() and json.loads(report.read_text()).get('status') == 'PASS':
        print(stage,'already complete:',report)
        return None
    pid_path=DRIVE_ROOT/'state'/f'{stage}.pid'
    script=next((part for part in command if part.endswith('.py')),None)
    if script:
        for line in subprocess.check_output(['ps','-eo','pid=,args='],text=True).splitlines():
            fields=line.strip().split(None,1)
            if len(fields)==2 and script in fields[1] and str(report.parent) in fields[1]:
                pid_path.write_text(fields[0])
                print(stage,'already running; pid=',fields[0],'log=',log_path)
                return int(fields[0])
    if pid_path.exists():
        old_pid=int(pid_path.read_text())
        alive=subprocess.run(['ps','-p',str(old_pid),'-o','args='],text=True,capture_output=True).stdout.strip()
        if alive and str(report) in alive:
            print(stage,'still running; pid=',old_pid,'log=',log_path)
            return old_pid
        pid_path.unlink()
    log_path.parent.mkdir(parents=True,exist_ok=True)
    env=os.environ.copy(); env['PYTHONPATH']=str(ROOT/'src'); env['PYTHONUNBUFFERED']='1'
    handle=log_path.open('a',encoding='utf-8')
    process=subprocess.Popen(command,cwd=ROOT,env=env,stdout=handle,stderr=subprocess.STDOUT,text=True,start_new_session=True)
    pid_path.write_text(str(process.pid))
    globals().setdefault('_stage_log_handles',[]).append(handle)
    print(stage,'started; pid=',process.pid,'log=',log_path)
    return process.pid

FROZEN_DIR=DRIVE_ROOT/'checkpoints'/'caduceus_frozen_head'
FROZEN_REPORT=FROZEN_DIR/'run.json'
FROZEN_CKPT=FROZEN_DIR/'latest.pt'
command=[PY,'scripts/adaptation/train_caduceus.py','--root',str(ROOT),'--reference',str(REFERENCE),'--cache-dir',str(DRIVE_ROOT/'model_cache'),'--output-dir',str(FROZEN_DIR),'--state',str(STATE),'--device','cuda','--stage','frozen_head_only','--epochs','3','--batch-size','1','--effective-batch-size','16','--dropout','0.1','--max-length','8192','--learning-rate','2e-5','--weight-decay','0.01','--seed','42']
if FROZEN_CKPT.exists(): command += ['--resume',str(FROZEN_CKPT)]
launch('caduceus_frozen_head',command,FROZEN_REPORT,DRIVE_ROOT/'logs'/'caduceus_frozen_head.log')


In [ ]:
def monitor(stage, report, log_path, checkpoint_dir=None):
    pid_path=DRIVE_ROOT/'state'/f'{stage}.pid'
    pid=int(pid_path.read_text()) if pid_path.exists() else None
    args=subprocess.run(['ps','-p',str(pid),'-o','etime=,pcpu=,stat=,args='],text=True,capture_output=True).stdout.strip() if pid else ''
    print('process=',args or 'not running')
    report=Path(report)
    print('report=',report.read_text() if report.exists() else 'PENDING')
    if checkpoint_dir:
        folder=Path(checkpoint_dir)
        print('checkpoints=',[(p.name,p.stat().st_size) for p in folder.glob('*')])
    log=Path(log_path)
    print('log tail=',log.read_text(errors='replace')[-4000:] if log.exists() else 'PENDING')

monitor('caduceus_frozen_head',FROZEN_REPORT,DRIVE_ROOT/'logs'/'caduceus_frozen_head.log',FROZEN_DIR)


## TRAIN-only grouped HPO

Start only after the frozen-head run report is persisted. The Optuna SQLite DB, per-trial/fold checkpoints, OOF predictions, and JSON progress report live on Drive. Rerunning resumes the same study; the selection lock appears only after at least eight completed trials.


In [ ]:
HPO = DRIVE_ROOT/'hpo'/'caduceus_hpo.json'
LOCK = DRIVE_ROOT/'hpo'/'selection_closed.json'
if not FROZEN_REPORT.exists() or json.loads(FROZEN_REPORT.read_text()).get('status') != 'PASS':
    print('Blocked: frozen-head baseline is not complete yet.')
elif LOCK.exists():
    print('Selection already closed:',LOCK.read_text())
else:
    command=[PY,'scripts/adaptation/run_caduceus_hpo.py','--root',str(ROOT),'--reference',str(REFERENCE),'--cache-dir',str(DRIVE_ROOT/'model_cache'),'--checkpoint-dir',str(DRIVE_ROOT/'checkpoints'/'caduceus_hpo'),'--state',str(STATE),'--output',str(HPO),'--device','cuda','--trials','8','--microbatch-size','2','--max-length','8192','--seed','42']
    launch('caduceus_hpo',command,HPO,DRIVE_ROOT/'logs'/'caduceus_hpo.log')


In [ ]:
monitor('caduceus_hpo',HPO,DRIVE_ROOT/'logs'/'caduceus_hpo.log',DRIVE_ROOT/'checkpoints'/'caduceus_hpo')
print('selection_lock=',LOCK.exists())


## Final refit and one-shot adaptation holdout

Create calibration and abstention thresholds from the selected trial's TRAIN OOF predictions before final refit. Do not run the holdout cell until TRAIN-only HPO is closed and final training has a valid checkpoint. The evaluator requires the hash-matched OOF calibration report and refuses a second 801-row output.


In [ ]:
if not LOCK.exists():
    print('Blocked: selection is still open; TRAIN OOF analysis waits for the lock.')
else:
    selection=json.loads(LOCK.read_text())
    OOF=Path(selection['train_oof_csv'])
    CALIBRATION_REPORT=DRIVE_ROOT/'analysis'/'caduceus_train_oof_calibration.json'
    if CALIBRATION_REPORT.exists():
        print('TRAIN OOF calibration report already exists; verify its OOF hash against the selection lock.')
    elif not OOF.exists():
        print('Blocked: selected TRAIN OOF predictions are missing.')
    else:
        subprocess.run([PY,'scripts/adaptation/run_posthoc_analysis.py','--root',str(ROOT),'--predictions',str(OOF),'--output',str(CALIBRATION_REPORT),'--state',str(STATE)],check=True)


In [ ]:
if not LOCK.exists():
    print('Blocked: selection is still open; VALIDATION remains closed.')
else:
    selection=json.loads(LOCK.read_text()); params=selection['selected_params']
    RUN=DRIVE_ROOT/'runs'/'caduceus_final'
    FINAL_REPORT=RUN/'run.json'; FINAL_CKPT=RUN/'latest.pt'
    command=[PY,'scripts/adaptation/train_caduceus.py','--root',str(ROOT),'--reference',str(REFERENCE),'--cache-dir',str(DRIVE_ROOT/'model_cache'),'--output-dir',str(RUN),'--state',str(STATE),'--selection-lock',str(LOCK),'--device','cuda','--stage',params['regime'],'--epochs',str(selection['final_epochs']),'--batch-size','1','--effective-batch-size',str(params['effective_batch_size']),'--dropout',str(params['dropout']),'--max-length','8192','--learning-rate',str(params['learning_rate']),'--weight-decay',str(params['weight_decay']),'--seed',str(selection['seed'])]
    if FINAL_CKPT.exists(): command += ['--resume',str(FINAL_CKPT)]
    launch('caduceus_final',command,FINAL_REPORT,DRIVE_ROOT/'logs'/'caduceus_final.log')


In [ ]:
if not LOCK.exists():
    print('Blocked: no closed TRAIN-only selection.')
else:
    RUN=DRIVE_ROOT/'runs'/'caduceus_final'; FINAL_CKPT=RUN/'latest.pt'
    HOLDOUT=DRIVE_ROOT/'exports'/'caduceus_validation.csv'
    CALIBRATION_REPORT=DRIVE_ROOT/'analysis'/'caduceus_train_oof_calibration.json'
    HOLDOUT_REPORT=HOLDOUT.with_suffix('.json')
    HOLDOUT_ATTEMPT=HOLDOUT.with_name(HOLDOUT.stem+'.attempt.json')
    if HOLDOUT.exists() or HOLDOUT_REPORT.exists() or HOLDOUT_ATTEMPT.exists():
        print('One-shot holdout artifact already exists; refusing a rerun.')
        print(HOLDOUT_REPORT.read_text() if HOLDOUT_REPORT.exists() else 'REPORT MISSING: reconcile before proceeding.')
    elif not (RUN/'run.json').exists() or not FINAL_CKPT.exists():
        print('Blocked: final TRAIN-only refit is incomplete.')
    else:
        if not CALIBRATION_REPORT.exists(): print('Blocked: matching TRAIN-OOF calibration report is missing.')
        else: subprocess.run([PY,'scripts/adaptation/evaluate_caduceus.py','--root',str(ROOT),'--reference',str(REFERENCE),'--cache-dir',str(DRIVE_ROOT/'model_cache'),'--checkpoint',str(FINAL_CKPT),'--selection-lock',str(LOCK),'--calibration-report',str(CALIBRATION_REPORT),'--state',str(STATE),'--output',str(HOLDOUT),'--split','validation','--device','cuda'],check=True)


In [ ]:
if STATE.exists():
    state=json.loads(STATE.read_text())
    print('latest state:',state.get('stage'))
    for event in state.get('history',[]): print(event['stage'],event['timestamp_utc'],event.get('details',{}))
else:
    print('No persisted state yet.')
for name in ('CADUCEUS_PARTIAL_DONE','CADUCEUS_FULL_DONE','NT_FROZEN_DONE','NT_PEFT_DONE','CALIBRATION_DONE','ABSTENTION_DONE','ENSEMBLE_DONE','ABLATIONS_DONE','CONTEXT_DONE','LEARNING_CURVES_DONE','STATISTICS_DONE','FIGURES_DONE','REPORT_DONE'):
    print(name,'PENDING until an evidence-backed runner writes the stage artifact')
